# Sinhala QA — Single-Question Answer (RAG context)

Retrieves context for one question, then generates an answer using the exact same
model/prompt/generation/grounding configuration as `qa-evaluation.ipynb` (v6-matched).

Requires `retrieve_context(query, top_k, mode)` and `knowledge_boundary_detection(query,
top_k)` to already be defined or imported (from your retrieval module/notebook) before
running the retrieval cell below — not implemented in this notebook.

In [ ]:
# Installs the packages needed to load and run the model.
%uv pip install -q "transformers>=4.51,<5" accelerate safetensors huggingface_hub hf_transfer

In [ ]:
# Configuration — model choice, generation/grounding settings (copied from
# qa-evaluation.ipynb, v6-matched), and the question to answer.
import os
import re
import unicodedata

import torch

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

MODEL_ID = "isji/sinllama-3b-qa-v6-merged"   # or "isji/sinllama-1b-qa-v6-merged"

NO_ANSWER = "මෙම ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත."
MAX_LENGTH = 2048
MAX_NEW_TOKENS = 48
REPETITION_PENALTY = 1.05
GROUNDING_THRESHOLD = 0.50
USE_GROUNDING = True

QUERY = "පරුමක යනු කවුරුන්ද?"
TOP_K = 5

In [ ]:
# Logs in to Hugging Face — only needed if MODEL_ID is a private repo. Reads the token from
# the environment; never paste a token directly into a cell.
from huggingface_hub import login

_token = os.environ.get("HF_TOKEN")
if _token:
    login(token=_token, add_to_git_credential=False)
    print("Logged in to Hugging Face.")
else:
    print("No HF_TOKEN set — continuing anonymously (fine for public repos).")

In [ ]:
# Loads the merged model and its tokenizer.
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=model_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()
model.config.pad_token_id = tokenizer.pad_token_id
print("Model loaded on", model.device)

In [ ]:
# Defines the v6 prompt template and the answer function: build the prompt, generate
# greedily, then apply the grounding gate (refuse if the answer isn't traceable to the
# context) — same logic and thresholds as qa-evaluation.ipynb.
SINHALA_WORD_RE = re.compile(r"[\w඀-෿]+", re.UNICODE)
STOPWORDS = {
    "හා", "සහ", "හෝ", "දී", "ද", "ය", "යි", "වේ", "විය", "වූ", "ලෙස",
    "විසින්", "සඳහා", "සිට", "දක්වා", "එම", "මෙම", "ඒ", "ඔහු", "ඇය",
    "කුමක්ද", "කවුද", "කවදාද", "කෙසේද", "කොපමණද", "මොනවාද",
}

INSTRUCTION = f"""උපදෙස්: පහත සන්දර්භය පමණක් භාවිතා කර ප්‍රශ්නයට පිළිතුරු දෙන්න.
- පිළිතුර සන්දර්භයේ තිබේ නම්, එයින් කෙටිම නිශ්චිත වචන පෙළ පමණක් දෙන්න.
- අමතර පැහැදිලි කිරීම්, පිටත දැනුම හෝ අනුමාන එකතු නොකරන්න.
- සන්දර්භය ප්‍රශ්නයට අදාළ නොවේ නම්, ප්‍රශ්නයට පිළිතුරු දීමට සුදුසු නොවේ නම්, හෝ පිළිතුර සන්දර්භයේ පැහැදිලිව නොමැති නම්, හරියටම මෙය පමණක් දෙන්න: {NO_ANSWER}"""


def clean_text(value):
    text = unicodedata.normalize("NFC", str(value or ""))
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()


def lexical_tokens(value):
    tokens = [t.casefold() for t in SINHALA_WORD_RE.findall(clean_text(value))]
    return [t for t in tokens if len(t) >= 2 and t not in STOPWORDS]


def normalize_answer(value):
    text = clean_text(value).casefold()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" \t\r\n[]{}()<>\"'`.,!?;:।෴")


def is_no_answer(value):
    normalized = normalize_answer(value)
    return normalized == normalize_answer(NO_ANSWER) or "ප්‍රමාණවත් තොරතුරු නොමැත" in normalized


def build_prompt(context, question):
    return (
        f"{INSTRUCTION}\n\n"
        f"සන්දර්භය:\n{clean_text(context)}\n\n"
        f"ප්‍රශ්නය:\n{clean_text(question)}\n\n"
        "පිළිතුර:\n"
    )


def evidence_support(answer, context):
    # Fraction of the answer's content words traceable to the context (a trailing
    # case-ending mismatch of one character is still counted as a match).
    answer_tokens = lexical_tokens(answer)
    if not answer_tokens:
        return 0.0
    normalized_context = " ".join(lexical_tokens(context))

    def supported(token):
        return token in normalized_context or (len(token) >= 4 and token[:-1] in normalized_context)

    return sum(supported(t) for t in answer_tokens) / len(answer_tokens)


def run_qa(context, question):
    prompt = build_prompt(context, question)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=REPETITION_PENALTY,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[-1]:]
    raw_answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    raw_answer = raw_answer.splitlines()[0].strip(" []{}()<>\"'`") if raw_answer else ""

    if not raw_answer or is_no_answer(raw_answer):
        return {"answer": NO_ANSWER, "raw_answer": raw_answer, "support": 1.0, "gated": False}

    support = evidence_support(raw_answer, context)
    if USE_GROUNDING and support < GROUNDING_THRESHOLD:
        return {"answer": NO_ANSWER, "raw_answer": raw_answer, "support": support, "gated": True}

    return {"answer": raw_answer, "raw_answer": raw_answer, "support": support, "gated": False}


print("run_qa ready.")

In [ ]:
# Retrieves context for QUERY (hybrid retrieval) and checks the syllabus-boundary
# classifier, then prints both. retrieve_context() and knowledge_boundary_detection() must
# already be defined/imported (see the notebook intro) before this cell runs.
results, context_str = retrieve_context(QUERY, top_k=TOP_K, mode="hybrid")
boundary = knowledge_boundary_detection(QUERY, top_k=TOP_K)

print(boundary["reason"])
print(f"Label : {boundary['label']}  (P={boundary['probability_within_syllabus']:.3f})")
print(f"Query : {QUERY}")
print(f"Chunks: {len(results)}")
for r in results:
    location = " · ".join(filter(None, [
        f"Grade {r['grade']}" if r.get("grade") else "",
        r.get("chapter"), r.get("section"),
        f"p.{r['page']}" if r.get("page") else "",
    ]))
    print(f"  [{r['rank']}] score={r['score']:.5f}  {location}")
    print(f"       {r['text'].replace(chr(10), ' ')}")

In [ ]:
# Generates and prints the final answer for QUERY, using the retrieved context above.
result = run_qa(context_str, QUERY)

print("Question :", QUERY)
print("Answer   :", result["answer"])
print(f"Support  : {result['support']:.3f}" + ("  (gated to refusal)" if result["gated"] else ""))